## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

# from resnet_20_32_44_56_v1 import *
from resnet_20_32_44_56_v2 import *
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=256, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Model
print('==> Building Model..\n')

net = ResNet20()
# net = ResNet32()
# net = ResNet44()
# net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
# 'We start with a learning rate of 0.1, divide it by 10 at 32k and 48k iterations, and terminate training at 64k iterations' (p.7)
# This code terminates training at the 200 epoch, so the learning rate is divided by 10 at the 100 and 150 epochs to match the same ratio as in the paper
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [6]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [7]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 244ms | Tot: 8s214ms | Loss: 1.800 | Acc: 32.660% (16330/50000) 196/196 
  Step: 10ms | Tot: 1s23ms | Loss: 1.680 | Acc: 37.380% (3738/10000) 100/100 
Saving..

Epoch: 1
  Step: 41ms | Tot: 8s70ms | Loss: 1.391 | Acc: 48.680% (24340/50000) 196/196  
  Step: 10ms | Tot: 1s41ms | Loss: 1.418 | Acc: 50.140% (5014/10000) 100/100 00 
Saving..

Epoch: 2
  Step: 41ms | Tot: 8s63ms | Loss: 1.118 | Acc: 60.036% (30018/50000) 196/196  
  Step: 10ms | Tot: 1s53ms | Loss: 1.193 | Acc: 59.380% (5938/10000) 100/100 
Saving..

Epoch: 3
  Step: 43ms | Tot: 8s79ms | Loss: 0.947 | Acc: 66.354% (33177/50000) 196/196  
  Step: 9ms | Tot: 1s92ms | Loss: 0.936 | Acc: 66.920% (6692/10000) 100/100 100 
Saving..

Epoch: 4
  Step: 41ms | Tot: 8s53ms | Loss: 0.828 | Acc: 70.846% (35423/50000) 196/196  
  Step: 10ms | Tot: 1s78ms | Loss: 1.035 | Acc: 65.750% (6575/10000) 100/100 100 

Epoch: 5
  Step: 42ms | Tot: 8s41ms | Loss: 0.732 | Acc: 74.202% (37101/50000) 196/196  /196 
  Step: 9ms | Tot:

  Step: 41ms | Tot: 7s914ms | Loss: 0.240 | Acc: 91.524% (45762/50000) 196/196  12/19 23/196 51/19 57/196 103/196 122/19 139/196 140/19 153/19 165/196 
  Step: 8ms | Tot: 967ms | Loss: 0.420 | Acc: 86.740% (8674/10000) 100/100 
Saving..

Epoch: 43
  Step: 43ms | Tot: 7s905ms | Loss: 0.241 | Acc: 91.606% (45803/50000) 196/196 6 21/196 22/196 46/19 51/19 64/19 73/196 94/196  97/196 103/196 134/196 137/196 149/196 174/19 189/196 191/196 
  Step: 9ms | Tot: 985ms | Loss: 0.469 | Acc: 85.030% (8503/10000) 100/100 2/10 88/100 

Epoch: 44
  Step: 44ms | Tot: 7s923ms | Loss: 0.236 | Acc: 91.848% (45924/50000) 196/196 4/19 88/196 89/19 96/196 103/196 133/196 143/19 145/19 146/196 151/19 163/196 172/196 
  Step: 9ms | Tot: 966ms | Loss: 0.585 | Acc: 83.100% (8310/10000) 100/100 

Epoch: 45
  Step: 42ms | Tot: 7s929ms | Loss: 0.228 | Acc: 91.932% (45966/50000) 196/196 4/196 83/19 143/196 
  Step: 9ms | Tot: 966ms | Loss: 0.661 | Acc: 81.670% (8167/10000) 100/100 

Epoch: 46
  Step: 41ms | Tot: 7s

  Step: 9ms | Tot: 984ms | Loss: 0.351 | Acc: 91.110% (9111/10000) 100/100 8/100 60/100 

Epoch: 124
  Step: 42ms | Tot: 7s982ms | Loss: 0.033 | Acc: 99.012% (49506/50000) 196/196  122/19 128/196 165/196 
  Step: 9ms | Tot: 978ms | Loss: 0.353 | Acc: 91.000% (9100/10000) 100/100 

Epoch: 125
  Step: 42ms | Tot: 8s19ms | Loss: 0.033 | Acc: 98.994% (49497/50000) 196/196  
  Step: 9ms | Tot: 993ms | Loss: 0.354 | Acc: 90.950% (9095/10000) 100/100 71/100 80/100 

Epoch: 126
  Step: 43ms | Tot: 8s32ms | Loss: 0.033 | Acc: 98.990% (49495/50000) 196/196  
  Step: 9ms | Tot: 984ms | Loss: 0.358 | Acc: 91.030% (9103/10000) 100/100 2/100 

Epoch: 127
  Step: 41ms | Tot: 8s35ms | Loss: 0.031 | Acc: 99.098% (49549/50000) 196/196  
  Step: 9ms | Tot: 980ms | Loss: 0.357 | Acc: 90.940% (9094/10000) 100/100 

Epoch: 128
  Step: 42ms | Tot: 8s25ms | Loss: 0.032 | Acc: 99.038% (49519/50000) 196/196  134/19 135/196 
  Step: 9ms | Tot: 992ms | Loss: 0.359 | Acc: 91.110% (9111/10000) 100/100 /100 87/100 


In [8]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 91.17
Error: 8.83
